In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_excel("Online Retail.xlsx")

df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# =========================
# 2. TIME SPLIT (IMPORTANT)
# =========================
cutoff_date = df['InvoiceDate'].max() - pd.Timedelta(days=90)

train_df = df[df['InvoiceDate'] <= cutoff_date]
future_df = df[df['InvoiceDate'] > cutoff_date]

# =========================
# 3. FEATURE ENGINEERING (PAST DATA ONLY)
# =========================
customer_features = train_df.groupby('CustomerID').agg({
    'TotalPrice': 'sum',
    'InvoiceNo': 'nunique',
    'InvoiceDate': 'max'
}).reset_index()

customer_features.rename(columns={
    'TotalPrice': 'TotalSpending',
    'InvoiceNo': 'TotalOrders',
    'InvoiceDate': 'LastPurchase'
}, inplace=True)

reference_date = train_df['InvoiceDate'].max()
customer_features['Recency'] = (reference_date - customer_features['LastPurchase']).dt.days

# =========================
# 4. DEFINE REAL CHURN (FUTURE ACTIVITY)
# =========================
future_activity = future_df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
future_activity.rename(columns={'InvoiceNo': 'FutureOrders'}, inplace=True)

customer_features = customer_features.merge(future_activity, on='CustomerID', how='left')
customer_features['FutureOrders'] = customer_features['FutureOrders'].fillna(0)

# If customer did NOT purchase in next 90 days → churn
customer_features['Churn'] = np.where(customer_features['FutureOrders'] == 0, 1, 0)

# =========================
# 5. MODEL TRAINING
# =========================
X = customer_features[['TotalSpending', 'TotalOrders', 'Recency']]
y = customer_features['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 6. EVALUATION (REAL)
# =========================
test_probs = model.predict_proba(X_test)[:,1]
roc_score = roc_auc_score(y_test, test_probs)

print("\nReal Model ROC-AUC:", round(roc_score,4))

# =========================
# 7. PREDICT ALL CUSTOMERS
# =========================
customer_features['Churn_Probability'] = model.predict_proba(X)[:,1]

# Risk Level
customer_features['Risk_Level'] = pd.cut(
    customer_features['Churn_Probability'],
    bins=[0,0.3,0.6,1],
    labels=['Low Risk 🟢','Medium Risk 🟡','High Risk 🔴']
)

# Segment
def segment(row):
    if row['TotalSpending'] > 2000:
        return "High Value Customer ⭐⭐"
    elif row['TotalSpending'] > 800:
        return "Medium Value Customer ⭐"
    else:
        return "Low Value Customer"

customer_features['Customer_Segment'] = customer_features.apply(segment, axis=1)

# Retention Score
customer_features['Retention_Score'] = (
    (1 - customer_features['Churn_Probability']) * 100
).round(0)

# Smart Discount (dynamic logic)
def discount_logic(p):
    if p > 0.8:
        return 25
    elif p > 0.6:
        return 15
    else:
        return 5

customer_features['Recommended_Discount'] = customer_features['Churn_Probability'].apply(discount_logic)

customer_features['Dynamic_Price'] = (
    customer_features['TotalSpending'] *
    (1 - customer_features['Recommended_Discount']/100)
).round(2)

# Offer Message
def offer_msg(p):
    if p > 0.8:
        return "Urgent retention campaign"
    elif p > 0.6:
        return "Special personalized offer"
    else:
        return "Standard loyalty offer"

customer_features['Offer_Message'] = customer_features['Churn_Probability'].apply(offer_msg)

# =========================
# 8. OPTIONAL FULL LIST
# =========================
if input("\nShow full churn risk list? (yes/no): ").lower() == 'yes':
    top = customer_features.sort_values(
        by='Churn_Probability',
        ascending=False
    ).head(30)

    print("\nTop 30 High Risk Customers:\n")
    print(top[['CustomerID','TotalSpending','TotalOrders','Recency',
               'Churn_Probability','Risk_Level','Customer_Segment',
               'Retention_Score','Recommended_Discount','Dynamic_Price',
               'Offer_Message']])

# =========================
# 9. INTERACTIVE DASHBOARD
# =========================
while True:
    try:
        cid = float(input("\nEnter Customer ID (0 to exit): "))
        if cid == 0:
            break

        if cid in customer_features['CustomerID'].values:
            row = customer_features[
                customer_features['CustomerID'] == cid
            ].iloc[0]

            print("\n========= AI CUSTOMER DASHBOARD =========")
            print("Customer ID:", int(row['CustomerID']))
            print("Total Spending:", round(row['TotalSpending'],2))
            print("Total Orders:", int(row['TotalOrders']))
            print("Recency:", int(row['Recency']))
            print("Churn Probability:", round(row['Churn_Probability'],3))
            print("Risk Level:", row['Risk_Level'])
            print("Customer Segment:", row['Customer_Segment'])
            print("Retention Score:", int(row['Retention_Score']),"/100")
            print("Recommended Discount:", row['Recommended_Discount'],"%")
            print("Dynamic Price:", row['Dynamic_Price'])
            print("Offer:", row['Offer_Message'])
            print("==========================================")
        else:
            print("Customer ID not found.")

    except:
        print("Invalid input.")

        # =========================




Real Model ROC-AUC: 0.7179



Show full churn risk list? (yes/no):  no

Enter Customer ID (0 to exit):  0


Model saved successfully!


In [2]:
import joblib
joblib.dump(model, "model.pkl")
print("Model saved successfully!")

Model saved successfully!
